In [ ]:
from diffusers import StableDiffusionPipeline
import torch
from matplotlib import pyplot as plt

In [ ]:
model_ckpt = "CompVis/stable-diffusion-v1-4"
sd_pipeline = StableDiffusionPipeline.from_pretrained(
    model_ckpt, torch_dtype=torch.float16
).to("cuda")

In [ ]:
prompts = [
    "a photo of an astronaut riding a horse on mars",
    "A high tech solarpunk utopia in the Amazon rainforest",
    "A pikachu fine dining with a view to the Eiffel Tower",
    "A mecha robot in a favela in expressionist style",
    "an insect robot preparing a delicious meal",
    "A small cabin on top of a snowy mountain in the style of Disney, artstation",
]

images = sd_pipeline(prompts, num_images_per_prompt=1, output_type="np").images

print(images.shape)
# (6, 512, 512, 3)

In [ ]:
from torchmetrics.functional.multimodal import clip_score
from functools import partial

clip_score_fn = partial(clip_score, model_name_or_path="openai/clip-vit-base-patch16")


def calculate_clip_score(images, prompts):
    images_int = (images * 255).astype("uint8")
    clip_score = clip_score_fn(
        torch.from_numpy(images_int).permute(0, 3, 1, 2), prompts
    ).detach()
    return round(float(clip_score), 4)


sd_clip_score = calculate_clip_score(images, prompts)
print(f"CLIP score: {sd_clip_score}")
# CLIP score: 35.7038

In [ ]:
for image in images:
    plt.imshow(image)

    plt.axis("off")
    plt.show()

In [ ]:
seed = 0
generator = torch.manual_seed(seed)

images_1_4 = sd_pipeline(
    prompts, num_images_per_prompt=1, generator=generator, output_type="np"
).images

In [ ]:
device = "cuda"

model_ckpt_1_5 = "runwayml/stable-diffusion-v1-5"
sd_pipeline_1_5 = StableDiffusionPipeline.from_pretrained(
    model_ckpt_1_5,
).to(device)

# images_1_5 = sd_pipeline_1_5(
#     prompts, num_images_per_prompt=1, generator=generator, output_type="np"
# ).images

In [ ]:
sd_clip_score_1_4 = calculate_clip_score(images_1_4, prompts)
print(f"CLIP Score with v-1-4: {sd_clip_score_1_4}")
# CLIP Score with v-1-4: 34.9102

# sd_clip_score_1_5 = calculate_clip_score(images_1_5, prompts)
# print(f"CLIP Score with v-1-5: {sd_clip_score_1_5}")
# # CLIP Score with v-1-5: 36.2137